# 02 — Tools & Strict Mode

In notebook 01 you wrote `TOOLS` and `EXECUTORS` as loose dicts. That falls apart fast: no input validation, no consistent return shape, and the LLM can hallucinate argument types (`{"a": "seven"}` when you asked for an integer).

**By the end of this notebook you will:**
1. Refactor tools into a `BaseTool` ABC with a uniform `ToolResult` return shape
2. Build a `ToolRegistry` that dispatches safely (never raises)
3. Turn on OpenAI's `strict: True` and watch it kill an entire class of bugs
4. See *why tool descriptions* are 60% of agent quality


## Setup

Load env vars **before** importing anything that constructs an OpenAI client at import / init time. `ReActAgent.__init__` eagerly creates `AsyncOpenAI()`, which reads `OPENAI_API_KEY` from the process env — if the env isn't loaded yet, the import crashes with `OpenAIError: Missing credentials`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env before continuing"
print("env loaded ·", "OPENAI_API_KEY: ✅")

## 1. The pieces, side by side

Three small files in `src/deepbrief/tools/`:
- `base.py` — `BaseTool` ABC, `ToolResult` Pydantic model
- `registry.py` — `ToolRegistry` (dispatch never raises)
- agent loop is in `src/deepbrief/agents/react.py`

Let's read the ABC first.

In [ ]:
# Inspect the contract
from deepbrief.tools.base import BaseTool, ToolResult
import inspect

print(inspect.getsource(ToolResult))

In [ ]:
print(inspect.getsource(BaseTool))

## 2. Build a real tool — `add_numbers`

We'll port the `add` tool from notebook 01 to the new shape. Three things to notice:

1. **`name`** is `verb_noun`, not `add` alone — easier for the LLM to disambiguate at scale
2. **`description`** has *what* + *when to call*
3. **`additionalProperties: False`** is required for `strict: True` to be accepted by the API

In [ ]:
from deepbrief.tools.base import BaseTool, ToolResult

class AddNumbersTool(BaseTool):
    name = "add_numbers"
    description = (
        "Add two integers and return the sum. "
        "Call this when the user asks for arithmetic addition. "
        "Do NOT call this for multiplication, division, or non-integer inputs."
    )
    parameters_schema = {
        "type": "object",
        "properties": {
            "a": {"type": "integer", "description": "First addend"},
            "b": {"type": "integer", "description": "Second addend"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    }

    async def execute(self, a: int, b: int) -> ToolResult:
        return ToolResult(
            tool_name=self.name,
            input_args={"a": a, "b": b},
            output={"sum": a + b},
            success=True,
        )

tool = AddNumbersTool()
print("Schema sent to LLM:")
import json
print(json.dumps(tool.to_openai_function(), indent=2))

## 3. Wire it up with the registry

In [ ]:
from deepbrief.tools.registry import ToolRegistry

registry = ToolRegistry()
registry.register(AddNumbersTool())
print("Registered:", registry.list_tools())

# Try executing — note we never get an exception, even with garbage input
good = await registry.execute("add_numbers", {"a": 3, "b": 4})
print("good:", good)

missing = await registry.execute("multiply_numbers", {"a": 3, "b": 4})
print("missing tool:", missing)

bad_args = await registry.execute("add_numbers", {"x": 3})
print("bad args:", bad_args)

Notice all three return a `ToolResult`. The LLM will read the `error` field and self-correct. **No exception bubbled up** — that's the contract.

## 4. Run the agent through the registry

`ReActAgent` lives in `src/deepbrief/agents/react.py`. It's the same loop from notebook 01, factored as a class.

In [ ]:
from deepbrief.agents.react import ReActAgent

agent = ReActAgent(
    registry=registry,
    system_prompt="You are a helpful calculator. Use the available tools.",
    max_steps=6,
)
result = await agent.run("What is 17 + 25, and also 100 + 200?")
print("answer:", result.answer)
print("steps:", result.steps, "  terminated_by:", result.terminated_by)
for entry in result.trace:
    print(f"  step {entry['step']}: {entry['tool_calls']}")

## 5. Why strict mode matters — break it on purpose

Without `strict: True`, the LLM might emit `{"a": "seventeen"}` (string instead of integer). You'd catch it in your validator and round-trip back to the LLM — wasted call.

With `strict: True`, OpenAI runs **constrained decoding**: at each token it only samples from tokens that keep the JSON valid against your schema. The args are correct on first emit.

Let's prove this. We'll build TWO versions of the same tool — one with strict, one without — and ask the model an ambiguous question that often produces type confusion.

In [ ]:
# Same tool but with strict mode disabled
non_strict_schema = tool.to_openai_function()
non_strict_schema["function"]["strict"] = False

from openai import AsyncOpenAI
client = AsyncOpenAI()

AMBIGUOUS = "Add seventeen and twenty-five for me."

# Helper to inspect the args the model emits
async def show_args(schema, label):
    r = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": AMBIGUOUS}],
        tools=[schema], tool_choice="required",
    )
    tc = r.choices[0].message.tool_calls[0]
    print(f"{label}: {tc.function.arguments}")

await show_args(tool.to_openai_function(), "strict=True ")
await show_args(non_strict_schema,        "strict=False")

On `gpt-4o-mini` you'll usually see both produce `{"a": 17, "b": 25}` — the model's strong enough that strict mode isn't *strictly* necessary. The point is: with strict on, it's **guaranteed**. On weaker open-source models or longer schemas, ~3-5% of calls without strict come back malformed.
Same tool, different descriptions. Watch which one the LLM actually picks correctly.

In [ ]:
# Two tools with overlapping concerns — only the description disambiguates them
class WeatherNowTool(BaseTool):
    name = "weather_now"
    description = "Get current weather for a city."   # ❌ vague — missing when/when-not
    parameters_schema = {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"], "additionalProperties": False,
    }
    async def execute(self, city):
        return ToolResult(tool_name=self.name, input_args={"city": city},
                          output={"temp_c": 14, "conditions": "cloudy"}, success=True)

class WeatherForecastTool(BaseTool):
    name = "weather_forecast"
    description = (
        "Get a multi-day weather forecast for a city. "
        "Call this when the user asks about weather more than 24h in the future. "
        "Do NOT call this for current/today weather — use weather_now for that."
    )
    parameters_schema = {
        "type": "object",
        "properties": {
            "city": {"type": "string"},
            "days": {"type": "integer", "description": "Number of days, 1-7"},
        },
        "required": ["city", "days"], "additionalProperties": False,
    }
    async def execute(self, city, days):
        return ToolResult(tool_name=self.name, input_args={"city": city, "days": days},
                          output={"forecast": [{"day": i, "temp_c": 15+i} for i in range(days)]},
                          success=True)

weather_registry = ToolRegistry()
weather_registry.register(WeatherNowTool())
weather_registry.register(WeatherForecastTool())

weather_agent = ReActAgent(
    registry=weather_registry,
    system_prompt="You are a travel assistant. Pick the right tool.",
    max_steps=4,
)

for q in [
    "What's the weather like in Paris right now?",
    "Will it rain in Tokyo over the next 3 days?",
]:
    r = await weather_agent.run(q)
    called = r.trace[0]['tool_calls']
    print(f"q={q!r}\n   → tool={called}")

The forecast tool's description includes *when to call* AND *when NOT to call* — that's what makes the LLM disambiguate cleanly. Without the negative case, you'd see `weather_now` called for future-weather questions ~30% of the time on `gpt-4o-mini`.

**The rule**: every tool description has three parts in this order:
1. What the tool does
2. WHEN to call it
3. WHEN NOT to call it

## 7. Self-check

1. Why does `ToolRegistry.execute()` never raise?
2. What's the JSON-Schema requirement that `strict: True` adds?
3. Name the three parts of a good tool description.
4. If you're stuck on a model that doesn't support strict mode (vLLM, DeepSeek), what's your fallback?

## What's next

Notebook **03** — **Termination & Cost**. We'll build a `LoopGuard` that detects literal repetition, and a `CostMeter` that hard-stops when a per-run budget is exceeded. This is the part of agent code that separates mid-level from senior engineers.
